In [4]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
import os
import sys

root_path = os.path.abspath("../..")
sys.path.append(root_path)

from algorithm.MADDPG_old import MADDPGTrainer, MADDPGTester
from network_env.network_env_v14 import NetworkEnvV14

### Directories

In [6]:
RESOURCE_PATH = "./configs/resource_config.json"
LOG_PATH = "./results"
DATA = "./data"

### Configurations

In [ ]:
frames_per_batch = 100
n_iter = 1000
min_replay_size = 10000
memory_size = 10000
n_optimizer_steps = 100
train_batch_size = 128
actor_lr = 1e-4
critic_lr = 1e-4
max_grad_norm = 0.5
gamma = 0.99
polyak_tau = 0.005

critic_configs = {
        "num_cells":256,
        "depth":2,
        "share_parameter": True,
        "centralized_critic": True
    }

actor_configs = {
        "num_cells":256,
        "depth":2,
        "share_parameter":True
    }

MAX_QUEUE_LENGTH = 5 #5 times capacity
PENALTY = -5
ALPHA = 1.0
BETA = 1000.0


### Experiment 1

- Single slice
- Constant Demand = 0.5
- (lambda, rho) = (0.5,0.5), (0.1,0.9), (0.9,0.1)

In [ ]:
import numpy as np
n_agent = 1
test_demand = 0.5
latency_pref =  np.array([0.5, 0.6, 0.9, 0.1])
energy_pref = 1-latency_pref


In [ ]:
RESOURCE_PATH = None

for idx, (lambda_, rho_) in enumerate(zip(latency_pref, energy_pref)):
    print(f'idx={idx}, lambda = {lambda_}, rho = {rho_}')
    
    log_path = os.path.join(LOG_PATH,f"0.{idx}")
    
    train_env = NetworkEnvV11(
        n_slices=n_agent,
        resource_path=RESOURCE_PATH,
        traffic_path=None,
        log_path= os.path.join(log_path,'train'),
        test_demand=test_demand,
        max_queue_length=MAX_QUEUE_LENGTH,
        penalty_reward=PENALTY,
        latency_preference=[lambda_],
        energy_preference=[rho_]
    )

    test_env = NetworkEnvV11(
        n_slices=n_agent,
        resource_path=RESOURCE_PATH,
        traffic_path=None,
        log_path=os.path.join(log_path,'test'),
        test_demand=test_demand,
        max_queue_length=MAX_QUEUE_LENGTH,
        penalty_reward=PENALTY,
        latency_preference=[lambda_],
        energy_preference=[rho_]
    )

    trainer = MADDPGTrainer(
        environment=train_env,
        n_agent=n_agent,
        frames_per_batch=frames_per_batch,
        n_iter=n_iter,
        min_replay_size=min_replay_size,
        memory_size=memory_size,
        n_optimizer_steps= n_optimizer_steps,
        train_batch_size=train_batch_size,
        actor_lr=actor_lr,
        critic_lr=critic_lr,
        max_grad_norm=max_grad_norm,
        polyak_tau=polyak_tau,
        gamma=gamma,
        critic_configs=critic_configs,
        actor_configs=actor_configs,
        save_path=log_path,
        seed = 0,
    )

    tester = MADDPGTester(
        environment=test_env,
        n_agent=n_agent,
        frames_per_batch=frames_per_batch,
        n_iter=10,
        min_replay_size=min_replay_size,
        memory_size=memory_size,
        n_optimizer_steps= n_optimizer_steps,
        train_batch_size=train_batch_size,
        actor_lr=actor_lr,
        critic_lr=critic_lr,
        max_grad_norm=max_grad_norm,
        polyak_tau=polyak_tau,
        gamma=gamma,
        critic_configs=critic_configs,
        actor_configs=actor_configs,
        load_path=log_path
    )
    
    trainer.train()
    tester.test()
    train_env.save_statistics()
    test_env.save_statistics()

In [ ]:
RESOURCE_PATH = "./configs/resource_config.json"

frames_per_batch = 100
n_iter = 100
min_replay_size = 1000
memory_size = 10000
n_optimizer_steps = 100
train_batch_size = 128
actor_lr = 1e-4
critic_lr = 1e-4
max_grad_norm = 0.5
gamma = 0.99
polyak_tau = 0.005


PENALTY = -5

critic_configs = {
        "num_cells":256,
        "depth":2,
        "share_parameter": True,
        "centralized_critic": True
    }

actor_configs = {
        "num_cells":256,
        "depth":2,
        "share_parameter":False
    }


for idx, (lambda_, rho_) in enumerate(zip(latency_pref, energy_pref)):
    print(f'idx={idx}, lambda = {lambda_}, rho = {rho_}')
    
    log_path = os.path.join(LOG_PATH,f"1.{idx}")
    
    train_env = NetworkEnvV11(
        n_slices=n_agent,
        resource_path=RESOURCE_PATH,
        traffic_path=None,
        log_path= os.path.join(log_path,'train'),
        test_demand=test_demand,
        max_queue_length=MAX_QUEUE_LENGTH,
        penalty_reward=PENALTY,
        latency_preference=[lambda_],
        energy_preference=[rho_],
        max_steps=1000
    )

    test_env = NetworkEnvV11(
        n_slices=n_agent,
        resource_path=RESOURCE_PATH,
        traffic_path=None,
        log_path=os.path.join(log_path,'test'),
        test_demand=test_demand,
        max_queue_length=MAX_QUEUE_LENGTH,
        penalty_reward=PENALTY,
        latency_preference=[lambda_],
        energy_preference=[rho_]
    )

    trainer = MADDPGTrainer(
        environment=train_env,
        n_agent=n_agent,
        frames_per_batch=frames_per_batch,
        n_iter=n_iter,
        min_replay_size=min_replay_size,
        memory_size=memory_size,
        n_optimizer_steps= n_optimizer_steps,
        train_batch_size=train_batch_size,
        actor_lr=actor_lr,
        critic_lr=critic_lr,
        max_grad_norm=max_grad_norm,
        polyak_tau=polyak_tau,
        gamma=gamma,
        critic_configs=critic_configs,
        actor_configs=actor_configs,
        save_path=log_path,
        seed = 0,
        noise_sigma = 0.5,
        
    )

    tester = MADDPGTester(
        environment=test_env,
        n_agent=n_agent,
        frames_per_batch=frames_per_batch,
        n_iter=10,
        min_replay_size=min_replay_size,
        memory_size=memory_size,
        n_optimizer_steps= n_optimizer_steps,
        train_batch_size=train_batch_size,
        actor_lr=actor_lr,
        critic_lr=critic_lr,
        max_grad_norm=max_grad_norm,
        polyak_tau=polyak_tau,
        gamma=gamma,
        critic_configs=critic_configs,
        actor_configs=actor_configs,
        load_path=log_path
    )
    
    trainer.train()
    tester.test()
    train_env.save_statistics()
    test_env.save_statistics()

### Experiment 2

- Single slice
- Constant Demand = 0.3,0.5,0.7
- (lambda, rho) = (0.5,0.5)

In [ ]:
n_agent = 1
test_demand = [0.3,0.5,0.7]
latency_pref = 0.5
energy_pref = 0.5

In [ ]:
for idx, demand_ in enumerate(test_demand):
    print(f'idx={idx}, demand = {demand_}')
    
    log_path = os.path.join(LOG_PATH,f"2.{idx}")
    
    train_env = NetworkEnvV11(
        n_slices=n_agent,
        resource_path=RESOURCE_PATH,
        traffic_path=None,
        log_path= os.path.join(log_path,'train'),
        test_demand=demand_,
        max_queue_length=MAX_QUEUE_LENGTH,
        penalty_reward=PENALTY,
        latency_preference=[latency_pref],
        energy_preference=[energy_pref]
    )

    test_env = NetworkEnvV11(
        n_slices=n_agent,
        resource_path=RESOURCE_PATH,
        traffic_path=None,
        log_path=os.path.join(log_path,'test'),
        test_demand=demand_,
        max_queue_length=MAX_QUEUE_LENGTH,
        penalty_reward=PENALTY,
        latency_preference=[latency_pref],
        energy_preference=[energy_pref]
    )

    trainer = MADDPGTrainer(
        environment=train_env,
        n_agent=n_agent,
        frames_per_batch=frames_per_batch,
        n_iter=n_iter,
        min_replay_size=min_replay_size,
        memory_size=memory_size,
        n_optimizer_steps= n_optimizer_steps,
        train_batch_size=train_batch_size,
        actor_lr=actor_lr,
        critic_lr=critic_lr,
        max_grad_norm=max_grad_norm,
        polyak_tau=polyak_tau,
        gamma=gamma,
        critic_configs=critic_configs,
        actor_configs=actor_configs,
        save_path=log_path,
        seed = 1,
    )

    tester = MADDPGTester(
        environment=test_env,
        n_agent=n_agent,
        frames_per_batch=frames_per_batch,
        n_iter=10,
        min_replay_size=min_replay_size,
        memory_size=memory_size,
        n_optimizer_steps= n_optimizer_steps,
        train_batch_size=train_batch_size,
        actor_lr=actor_lr,
        critic_lr=critic_lr,
        max_grad_norm=max_grad_norm,
        polyak_tau=polyak_tau,
        gamma=gamma,
        critic_configs=critic_configs,
        actor_configs=actor_configs,
        load_path=log_path
    )
    
    trainer.train()
    tester.test()
    train_env.save_statistics()
    test_env.save_statistics()

### Experiment 3

- Single slice
- Uniform demand [0.3, 0.7]
- (lambda, rho) = (0.5,0.5)

In [ ]:
lambda_ = 0.5
rho_ = 0.5

traffic_path_train = os.path.join(DATA,"train")
traffic_path_test = os.path.join(DATA,"test")

In [ ]:
log_path = os.path.join(LOG_PATH,f"3")

train_env = NetworkEnvV11(
    n_slices=n_agent,
    resource_path=RESOURCE_PATH,
    traffic_path=traffic_path_train,
    log_path= os.path.join(log_path,'train'),
    test_demand= None,
    max_queue_length=MAX_QUEUE_LENGTH,
    penalty_reward=PENALTY,
    latency_preference=[lambda_],
    energy_preference=[rho_]
)

test_env = NetworkEnvV11(
    n_slices=n_agent,
    resource_path=RESOURCE_PATH,
    traffic_path=traffic_path_test,
    log_path=os.path.join(log_path,'test'),
    test_demand=None,
    max_queue_length=MAX_QUEUE_LENGTH,
    penalty_reward=PENALTY,
    latency_preference=[lambda_],
    energy_preference=[rho_]
)

trainer = MADDPGTrainer(
    environment=train_env,
    n_agent=n_agent,
    frames_per_batch=frames_per_batch,
    n_iter=n_iter,
    min_replay_size=min_replay_size,
    memory_size=memory_size,
    n_optimizer_steps= n_optimizer_steps,
    train_batch_size=train_batch_size,
    actor_lr=actor_lr,
    critic_lr=critic_lr,
    max_grad_norm=max_grad_norm,
    polyak_tau=polyak_tau,
    gamma=gamma,
    critic_configs=critic_configs,
    actor_configs=actor_configs,
    save_path=log_path,
    seed = 0,
)

tester = MADDPGTester(
    environment=test_env,
    n_agent=n_agent,
    frames_per_batch=frames_per_batch,
    n_iter=10,
    min_replay_size=min_replay_size,
    memory_size=memory_size,
    n_optimizer_steps= n_optimizer_steps,
    train_batch_size=train_batch_size,
    actor_lr=actor_lr,
    critic_lr=critic_lr,
    max_grad_norm=max_grad_norm,
    polyak_tau=polyak_tau,
    gamma=gamma,
    critic_configs=critic_configs,
    actor_configs=actor_configs,
    load_path=log_path
)

trainer.train()
tester.test()
train_env.save_statistics()
test_env.save_statistics()

### Experiment 4

- 3 slices
- Uniform demand [0.3, 0.7]
- latency_pref = [0.5, 0.1, 0.9]
- energy_pref = [0.5, 0.9, 0.1]

In [ ]:
lambda_ = [0.5,0.9,0.7]
rho_ = [0.5, 0.1, 0.3]

n_agent = 3

traffic_path_train = os.path.join(DATA,"train")
traffic_path_test = os.path.join(DATA,"test")

frames_per_batch = 100
n_iter = 100
min_replay_size = 1000
memory_size = 10000
n_optimizer_steps = 100
train_batch_size = 128
actor_lr = 1e-4
critic_lr = 1e-4
max_grad_norm = 1
gamma = 0.99
polyak_tau = 0.005

PENALTY = -5


In [ ]:
log_path = os.path.join(LOG_PATH,f"4")

train_env = NetworkEnvV11(
    n_slices=n_agent,
    resource_path=RESOURCE_PATH,
    traffic_path=traffic_path_train,
    log_path= os.path.join(log_path,'train'),
    test_demand= None,
    max_queue_length=MAX_QUEUE_LENGTH,
    penalty_reward=PENALTY,
    latency_preference=lambda_,
    energy_preference=rho_
)

test_env = NetworkEnvV11(
    n_slices=n_agent,
    resource_path=RESOURCE_PATH,
    traffic_path=traffic_path_test,
    log_path=os.path.join(log_path,'test'),
    test_demand=None,
    max_queue_length=MAX_QUEUE_LENGTH,
    penalty_reward=PENALTY,
    latency_preference=lambda_,
    energy_preference=rho_
)



trainer = MADDPGTrainer(
    environment=train_env,
    n_agent=n_agent,
    frames_per_batch=frames_per_batch,
    n_iter=n_iter,
    min_replay_size=min_replay_size,
    memory_size=memory_size,
    n_optimizer_steps= n_optimizer_steps,
    train_batch_size=train_batch_size,
    actor_lr=actor_lr,
    critic_lr=critic_lr,
    max_grad_norm=max_grad_norm,
    polyak_tau=polyak_tau,
    gamma=gamma,
    critic_configs=critic_configs,
    actor_configs=actor_configs,
    save_path=log_path,
    seed = 0,
)

tester = MADDPGTester(
    environment=test_env,
    n_agent=n_agent,
    frames_per_batch=frames_per_batch,
    n_iter=10,
    min_replay_size=min_replay_size,
    memory_size=memory_size,
    n_optimizer_steps= n_optimizer_steps,
    train_batch_size=train_batch_size,
    actor_lr=actor_lr,
    critic_lr=critic_lr,
    max_grad_norm=max_grad_norm,
    polyak_tau=polyak_tau,
    gamma=gamma,
    critic_configs=critic_configs,
    actor_configs=actor_configs,
    load_path=log_path
)

trainer.train()
tester.test()
train_env.save_statistics()
test_env.save_statistics()

## Experiment 5

In [ ]:
import numpy as np
n_agent = 1
test_demand = 0.5
latency_pref =  np.array([0.5, 0.6, 0.9, 0.1])
energy_pref = 1-latency_pref


RESOURCE_PATH = None
traffic_path_train = os.path.join(DATA,"train")
traffic_path_test = os.path.join(DATA,"test")

for idx, (lambda_, rho_) in enumerate(zip(latency_pref, energy_pref)):
    print(f'idx={idx}, lambda = {lambda_}, rho = {rho_}')
    
    log_path = os.path.join(LOG_PATH,f"5.{idx}")
    
    train_env = NetworkEnvV11(
        n_slices=n_agent,
        resource_path=RESOURCE_PATH,
        traffic_path=traffic_path_train,
        log_path= os.path.join(log_path,'train'),
        test_demand=None,
        max_queue_length=MAX_QUEUE_LENGTH,
        penalty_reward=PENALTY,
        latency_preference=[lambda_],
        energy_preference=[rho_]
    )

    test_env = NetworkEnvV11(
        n_slices=n_agent,
        resource_path=RESOURCE_PATH,
        traffic_path=traffic_path_test,
        log_path=os.path.join(log_path,'test'),
        test_demand=None,
        max_queue_length=MAX_QUEUE_LENGTH,
        penalty_reward=PENALTY,
        latency_preference=[lambda_],
        energy_preference=[rho_]
    )

    trainer = MADDPGTrainer(
        environment=train_env,
        n_agent=n_agent,
        frames_per_batch=frames_per_batch,
        n_iter=n_iter,
        min_replay_size=min_replay_size,
        memory_size=memory_size,
        n_optimizer_steps= n_optimizer_steps,
        train_batch_size=train_batch_size,
        actor_lr=actor_lr,
        critic_lr=critic_lr,
        max_grad_norm=max_grad_norm,
        polyak_tau=polyak_tau,
        gamma=gamma,
        critic_configs=critic_configs,
        actor_configs=actor_configs,
        save_path=log_path,
        seed = 0,
    )

    tester = MADDPGTester(
        environment=test_env,
        n_agent=n_agent,
        frames_per_batch=frames_per_batch,
        n_iter=10,
        min_replay_size=min_replay_size,
        memory_size=memory_size,
        n_optimizer_steps= n_optimizer_steps,
        train_batch_size=train_batch_size,
        actor_lr=actor_lr,
        critic_lr=critic_lr,
        max_grad_norm=max_grad_norm,
        polyak_tau=polyak_tau,
        gamma=gamma,
        critic_configs=critic_configs,
        actor_configs=actor_configs,
        load_path=log_path
    )
    
    trainer.train()
    tester.test()
    train_env.save_statistics()
    test_env.save_statistics()

## Experiment 6

In [ ]:
import numpy as np
n_agent = 1
test_demand = 0.5
latency_pref =  10 * np.array([0.5, 0.6, 0.9, 0.1])
energy_pref = 10-latency_pref

PENALTY = -50

n_iter = 100



RESOURCE_PATH = "./configs/resource_config.json"
traffic_path_train = os.path.join(DATA,"train")
traffic_path_test = os.path.join(DATA,"test")

for idx, (lambda_, rho_) in enumerate(zip(latency_pref, energy_pref)):
    print(f'idx={idx}, lambda = {lambda_}, rho = {rho_}')
    
    log_path = os.path.join(LOG_PATH,f"6.{idx}")
    
    train_env = NetworkEnvV11(
        n_slices=n_agent,
        resource_path=RESOURCE_PATH,
        traffic_path=traffic_path_train,
        log_path= os.path.join(log_path,'train'),
        test_demand=None,
        max_queue_length=MAX_QUEUE_LENGTH,
        penalty_reward=PENALTY,
        latency_preference=[lambda_],
        energy_preference=[rho_]
    )

    test_env = NetworkEnvV11(
        n_slices=n_agent,
        resource_path=RESOURCE_PATH,
        traffic_path=traffic_path_test,
        log_path=os.path.join(log_path,'test'),
        test_demand=None,
        max_queue_length=MAX_QUEUE_LENGTH,
        penalty_reward=PENALTY,
        latency_preference=[lambda_],
        energy_preference=[rho_]
    )

    trainer = MADDPGTrainer(
        environment=train_env,
        n_agent=n_agent,
        frames_per_batch=frames_per_batch,
        n_iter=n_iter,
        min_replay_size=min_replay_size,
        memory_size=memory_size,
        n_optimizer_steps= n_optimizer_steps,
        train_batch_size=train_batch_size,
        actor_lr=actor_lr,
        critic_lr=critic_lr,
        max_grad_norm=max_grad_norm,
        polyak_tau=polyak_tau,
        gamma=gamma,
        critic_configs=critic_configs,
        actor_configs=actor_configs,
        save_path=log_path,
        seed = 0,
    )

    tester = MADDPGTester(
        environment=test_env,
        n_agent=n_agent,
        frames_per_batch=frames_per_batch,
        n_iter=10,
        min_replay_size=min_replay_size,
        memory_size=memory_size,
        n_optimizer_steps= n_optimizer_steps,
        train_batch_size=train_batch_size,
        actor_lr=actor_lr,
        critic_lr=critic_lr,
        max_grad_norm=max_grad_norm,
        polyak_tau=polyak_tau,
        gamma=gamma,
        critic_configs=critic_configs,
        actor_configs=actor_configs,
        load_path=log_path
    )
    
    trainer.train()
    tester.test()
    train_env.save_statistics()
    test_env.save_statistics()

In [ ]:
import numpy as np
n_agent = 1
test_demand = 0.5
latency_pref =  10 * np.array([0.5, 0.6, 0.9, 0.1])
energy_pref = 10-latency_pref

PENALTY = -50

n_iter = 100

#Also multipy stable queue by 10

RESOURCE_PATH = "./configs/resource_config.json"
traffic_path_train = os.path.join(DATA,"train")
traffic_path_test = os.path.join(DATA,"test")

for idx, (lambda_, rho_) in enumerate(zip(latency_pref, energy_pref)):
    print(f'idx={idx}, lambda = {lambda_}, rho = {rho_}')
    
    log_path = os.path.join(LOG_PATH,f"7.{idx}")
    
    train_env = NetworkEnvV11(
        n_slices=n_agent,
        resource_path=RESOURCE_PATH,
        traffic_path=traffic_path_train,
        log_path= os.path.join(log_path,'train'),
        test_demand=None,
        max_queue_length=MAX_QUEUE_LENGTH,
        penalty_reward=PENALTY,
        latency_preference=[lambda_],
        energy_preference=[rho_]
    )

    test_env = NetworkEnvV11(
        n_slices=n_agent,
        resource_path=RESOURCE_PATH,
        traffic_path=traffic_path_test,
        log_path=os.path.join(log_path,'test'),
        test_demand=None,
        max_queue_length=MAX_QUEUE_LENGTH,
        penalty_reward=PENALTY,
        latency_preference=[lambda_],
        energy_preference=[rho_]
    )

    trainer = MADDPGTrainer(
        environment=train_env,
        n_agent=n_agent,
        frames_per_batch=frames_per_batch,
        n_iter=n_iter,
        min_replay_size=min_replay_size,
        memory_size=memory_size,
        n_optimizer_steps= n_optimizer_steps,
        train_batch_size=train_batch_size,
        actor_lr=actor_lr,
        critic_lr=critic_lr,
        max_grad_norm=max_grad_norm,
        polyak_tau=polyak_tau,
        gamma=gamma,
        critic_configs=critic_configs,
        actor_configs=actor_configs,
        save_path=log_path,
        seed = 0,
    )

    tester = MADDPGTester(
        environment=test_env,
        n_agent=n_agent,
        frames_per_batch=frames_per_batch,
        n_iter=10,
        min_replay_size=min_replay_size,
        memory_size=memory_size,
        n_optimizer_steps= n_optimizer_steps,
        train_batch_size=train_batch_size,
        actor_lr=actor_lr,
        critic_lr=critic_lr,
        max_grad_norm=max_grad_norm,
        polyak_tau=polyak_tau,
        gamma=gamma,
        critic_configs=critic_configs,
        actor_configs=actor_configs,
        load_path=log_path
    )
    
    trainer.train()
    tester.test()
    train_env.save_statistics()
    test_env.save_statistics()

## Experiments 9

In [ ]:
import numpy as np

n_agent = 3 
latency_pref =  np.array([1, 1, 1])
energy_pref = np.array([2, 2, 2])

PENALTY = -10 

uniform_demand = {'slice_0': {'low': 0.1, 'high': 0.2}, 'slice_1': {'low': 0.1, 'high': 0.2}, 'slice_2': {'low': 0.1, 'high': 0.2}}

In [ ]:
for i in range(10):
    print(f'idx={idx}, lambda = {lambda_}, rho = {rho_}')
    
    log_path = os.path.join(LOG_PATH,f"9.{idx}")
    
    train_env = NetworkEnvV14(
        n_slices=n_agent,
        resource_path=RESOURCE_PATH,
        traffic_path=None,
        log_path= os.path.join(log_path,'train'),
        test_demand=None,
        uniform_demand=uniform_demand,
        max_queue_length=MAX_QUEUE_LENGTH,
        penalty_reward=PENALTY,
        latency_preference=latency_pref,
        energy_preference=energy_pref,
        seed = i
    )

    test_env = NetworkEnvV14(
        n_slices=n_agent,
        resource_path=RESOURCE_PATH,
        traffic_path=None,
        log_path=os.path.join(log_path,'test'),
        test_demand=None,
        uniform_demand=uniform_demand,
        max_queue_length=MAX_QUEUE_LENGTH,
        penalty_reward=PENALTY,
        latency_preference=latency_pref,
        energy_preference=energy_pref,
        seed = 1
    )

    trainer = MADDPGTrainer(
        environment=train_env,
        n_agent=n_agent,
        frames_per_batch=frames_per_batch,
        n_iter=n_iter,
        min_replay_size=min_replay_size,
        memory_size=memory_size,
        n_optimizer_steps= n_optimizer_steps,
        train_batch_size=train_batch_size,
        actor_lr=actor_lr,
        critic_lr=critic_lr,
        max_grad_norm=max_grad_norm,
        polyak_tau=polyak_tau,
        gamma=gamma,
        critic_configs=critic_configs,
        actor_configs=actor_configs,
        save_path=log_path,
        seed = 0,
    )

    tester = MADDPGTester(
        environment=test_env,
        n_agent=n_agent,
        frames_per_batch=frames_per_batch,
        n_iter=10,
        min_replay_size=min_replay_size,
        memory_size=memory_size,
        n_optimizer_steps= n_optimizer_steps,
        train_batch_size=train_batch_size,
        actor_lr=actor_lr,
        critic_lr=critic_lr,
        max_grad_norm=max_grad_norm,
        polyak_tau=polyak_tau,
        gamma=gamma,
        critic_configs=critic_configs,
        actor_configs=actor_configs,
        load_path=log_path
    )
    
    trainer.train()
    tester.test()
    train_env.save_statistics()
    test_env.save_statistics()

In [ ]:
for i in range(10):
    
    log_path = os.path.join(LOG_PATH,f"10.{i}")
    
    train_env = NetworkEnvV14(
        n_slices=n_agent,
        resource_path=RESOURCE_PATH,
        traffic_path=None,
        log_path= os.path.join(log_path,'train'),
        test_demand=None,
        uniform_demand=uniform_demand,
        max_queue_length=MAX_QUEUE_LENGTH,
        penalty_reward=PENALTY,
        latency_preference=latency_pref,
        energy_preference=energy_pref,
        seed = i,
        safe=True
    )

    test_env = NetworkEnvV14(
        n_slices=n_agent,
        resource_path=RESOURCE_PATH,
        traffic_path=None,
        log_path=os.path.join(log_path,'test'),
        test_demand=None,
        uniform_demand=uniform_demand,
        max_queue_length=MAX_QUEUE_LENGTH,
        penalty_reward=PENALTY,
        latency_preference=latency_pref,
        energy_preference=energy_pref,
        seed = 1,
        safe = True
    )

    trainer = MADDPGTrainer(
        environment=train_env,
        n_agent=n_agent,
        frames_per_batch=frames_per_batch,
        n_iter=n_iter,
        min_replay_size=min_replay_size,
        memory_size=memory_size,
        n_optimizer_steps= n_optimizer_steps,
        train_batch_size=train_batch_size,
        actor_lr=actor_lr,
        critic_lr=critic_lr,
        max_grad_norm=max_grad_norm,
        polyak_tau=polyak_tau,
        gamma=gamma,
        critic_configs=critic_configs,
        actor_configs=actor_configs,
        save_path=log_path,
        seed = 0,
    )

    tester = MADDPGTester(
        environment=test_env,
        n_agent=n_agent,
        frames_per_batch=frames_per_batch,
        n_iter=10,
        min_replay_size=min_replay_size,
        memory_size=memory_size,
        n_optimizer_steps= n_optimizer_steps,
        train_batch_size=train_batch_size,
        actor_lr=actor_lr,
        critic_lr=critic_lr,
        max_grad_norm=max_grad_norm,
        polyak_tau=polyak_tau,
        gamma=gamma,
        critic_configs=critic_configs,
        actor_configs=actor_configs,
        load_path=log_path
    )
    
    trainer.train()
    tester.test()
    train_env.save_statistics()
    test_env.save_statistics()

In [1]:
import numpy as np


def random_policy(env):
    return {
        agent: env.action_space(agent).sample()
        for agent in env.agents
    }


def over_allocation_policy(env):
    return {
        agent: np.ones(
            env.action_space(agent).shape,
            dtype=np.float32,
        )
        for agent in env.agents
    }


def evaluate_policy(
    env,
    policy_fn,
    total_steps=100_000,
):
    obs, infos = env.reset()

    step_count = 0
    episode_count = 0

    while step_count < total_steps:

        actions = policy_fn(env)

        (
            obs,
            rewards,
            terminations,
            truncations,
            infos,
        ) = env.step(actions)

        step_count += 1

        done = any(
            terminations[a] or truncations[a]
            for a in env.agents
        )

        if done:

            episode_count += 1

            obs, infos = env.reset()

        if step_count % 1000 == 0:
            print(
                f"Steps: {step_count:,}/{total_steps:,}"
            )

    print(
        f"Finished {step_count:,} steps "
        f"({episode_count} episodes)"
    )

    return env